In [0]:
display(spark.sql("SHOW TABLES IN default"))

database,tableName,isTemporary
default,fundamentals,false
default,prices,false
default,securities,false


In [0]:
# ── Celda 2: Exploración inicial ───────────────────────────────
prices = spark.table("prices")
print(f"Filas: {prices.count():,}")
print(f"Columnas: {len(prices.columns)}")
prices.printSchema()

Filas: 851,264
Columnas: 7
root
 |-- date: timestamp (nullable = true)
 |-- symbol: string (nullable = true)
 |-- open: double (nullable = true)
 |-- close: double (nullable = true)
 |-- low: double (nullable = true)
 |-- high: double (nullable = true)
 |-- volume: double (nullable = true)



In [0]:
# ── Celda 3: Vista previa de los datos ────────────────────────
display(prices.limit(10))


date,symbol,open,close,low,high,volume
2016-01-05T00:00:00.000Z,WLTW,123.43,125.839996,122.309998,126.25,2163600.0
2016-01-06T00:00:00.000Z,WLTW,125.239998,119.980003,119.940002,125.540001,2386400.0
2016-01-07T00:00:00.000Z,WLTW,116.379997,114.949997,114.93,119.739998,2489500.0
2016-01-08T00:00:00.000Z,WLTW,115.480003,116.620003,113.5,117.440002,2006300.0
2016-01-11T00:00:00.000Z,WLTW,117.010002,114.970001,114.089996,117.330002,1408600.0
2016-01-12T00:00:00.000Z,WLTW,115.510002,115.550003,114.5,116.059998,1098000.0
2016-01-13T00:00:00.000Z,WLTW,116.459999,112.849998,112.589996,117.07,949600.0
2016-01-14T00:00:00.000Z,WLTW,113.510002,114.379997,110.050003,115.029999,785300.0
2016-01-15T00:00:00.000Z,WLTW,113.330002,112.529999,111.919998,114.879997,1093700.0
2016-01-19T00:00:00.000Z,WLTW,113.660004,110.379997,109.870003,115.870003,1523500.0


In [0]:
# ── Celda 4: Limpieza ─────────────────────────────────────────
from pyspark.sql.functions import col, to_date

# Convertir date a tipo fecha y eliminar nulos
prices_clean = prices \
    .withColumn("date", to_date(col("date"))) \
    .dropna()

print(f"Filas originales : {prices.count():,}")
print(f"Filas limpias    : {prices_clean.count():,}")

Filas originales : 851,264
Filas limpias    : 851,264


In [0]:
# ── Celda 5: Top 10 empresas por volumen de trading ───────────
from pyspark.sql.functions import avg, round as spark_round

top_volume = prices_clean \
    .groupBy("symbol") \
    .agg(spark_round(avg("volume"), 0).alias("avg_volume")) \
    .orderBy("avg_volume", ascending=False) \
    .limit(10)

display(top_volume)

symbol,avg_volume
BAC,1.4238697E8
AAPL,9.4225776E7
F,4.9352281E7
GE,4.8551661E7
MSFT,4.5797837E7
INTC,4.2045504E7
CSCO,4.198613E7
FB,4.0007884E7
PFE,3.7193774E7
HPQ,3.5795878E7


In [0]:
# ── Celda 6: Top 10 empresas más volátiles ────────────────────
from pyspark.sql.functions import avg, round as spark_round

volatility = prices_clean \
    .withColumn("daily_range", col("high") - col("low")) \
    .groupBy("symbol") \
    .agg(spark_round(avg("daily_range"), 2).alias("avg_volatility")) \
    .orderBy("avg_volatility", ascending=False) \
    .limit(10)

display(volatility)

symbol,avg_volatility
PCLN,18.56
GOOGL,11.52
GOOG,11.33
ISRG,10.08
CMG,9.27
REGN,7.78
AZO,7.7
AMZN,7.49
NFLX,6.84
AAPL,6.08


In [0]:
# ── Celda 7: Rentabilidad anual por empresa ───────────────────
from pyspark.sql.functions import year, first, last, round as spark_round

annual_return = prices_clean \
    .withColumn("year", year(col("date"))) \
    .groupBy("symbol", "year") \
    .agg(
        first("open").alias("price_start"),
        last("close").alias("price_end")
    ) \
    .withColumn("return_pct", spark_round((col("price_end") - col("price_start")) / col("price_start") * 100, 2)) \
    .groupBy("symbol") \
    .agg(spark_round(avg("return_pct"), 2).alias("avg_annual_return")) \
    .orderBy("avg_annual_return", ascending=False) \
    .limit(10)

display(annual_return)


symbol,avg_annual_return
LVLT,248.05
ETFC,133.78
C,82.96
MSI,69.45
NFLX,59.2
REGN,58.96
AAL,57.95
ULTA,47.03
FB,46.87
URI,46.31


In [0]:
# ── Celda 8: Volumen medio de trading por sector ──────────────
securities = spark.table("securities")

sector_volume = prices_clean \
    .join(securities, prices_clean.symbol == securities["Ticker symbol"], "inner") \
    .groupBy("GICS Sector") \
    .agg(spark_round(avg("volume"), 0).alias("avg_volume")) \
    .orderBy("avg_volume", ascending=False)

display(sector_volume)

GICS Sector,avg_volume
Telecommunications Services,1.3788971E7
Information Technology,1.0359266E7
Financials,7749111.0
Energy,5918965.0
Consumer Discretionary,4509139.0
Consumer Staples,4507772.0
Health Care,4069756.0
Materials,3747326.0
Industrials,3534210.0
Utilities,2815280.0


Databricks visualization. Run in Databricks to view.

In [0]:
# ── Celda 9: Resumen del análisis ─────────────────────────────
print("""
NYSE Stock Analysis — Apache Spark + Databricks
================================================
Dataset : 851,264 registros | 500 empresas | NYSE

Resultados:
  Top volumen      → BAC (142M acciones/día), AAPL (94M)
  Más volátiles    → PCLN ($18 rango diario), GOOGL ($11)
  Mayor rentabilidad → LVLT (+248% anual), NFLX (+59%)
  Sector más activo → Telecomunicaciones > IT > Financials
""")


NYSE Stock Analysis — Apache Spark + Databricks
Dataset : 851,264 registros | 500 empresas | NYSE

Resultados:
  Top volumen      → BAC (142M acciones/día), AAPL (94M)
  Más volátiles    → PCLN ($18 rango diario), GOOGL ($11)
  Mayor rentabilidad → LVLT (+248% anual), NFLX (+59%)
  Sector más activo → Telecomunicaciones > IT > Financials

